# Lamoda · повторяющиеся последовательности категорий

Прогнозируем покупку за следующие три месяца. Для каждого клиента строим последовательность основных категорий заказов и проверяем, когда похожая последовательность уже встречалась. Итоговый CSV записывается в `submission_motifs.csv`. Запускайте ноутбук из корня репозитория. Путь к исходному ZIP или каталогу данных можно задать в `LAMODA_DATA`.

In [1]:
%run ./lamoda_groups.ipynb

               size     mean
scoring_date                
2022-01-01    25000  0.34764
2022-02-01    25000  0.33368
2022-03-01    25000  0.33924


Матрицы признаков: (75000, 29) (25000, 29)


699 признаков


Переходы и все признаки: (75000, 733) (75000, 861)


date                  2022-02-01  2022-03-01     mean
model                                                
combined_regional        0.76002     0.77011  0.76506
full_regional            0.75932     0.76985  0.76458
transitions_regional     0.75930     0.76812  0.76371
transitions              0.75465     0.76282  0.75874
regional                 0.74834     0.76359  0.75597
Выбрано по средней AP@20%: combined_regional
Проверены 25,000 строк и порядок клиентов по образцу Kaggle.


### История заказов на дату прогноза

В предыдущем ноутбуке рассчитаны 861 исходный признак и прогноз для сравнения. Новые признаки измеряют длительность и частоту повторов последовательностей от одной до четырёх категорий, а также описывают покупки после прежних совпадений. Последний доступный день истории — за 14 дней до даты скоринга.

In [2]:
from research.motif_model import build_motifs, motif_experts
from research.window_models import forecast

core_train = full_train.set_axis(train.customer_id)
core_test = full_test.set_axis(test.customer_id)
cohorts = pd.concat([train[['customer_id','scoring_date']], test], ignore_index=True)
extra_motifs = build_motifs(orders, cohorts)
assert extra_motifs.index.is_unique and extra_motifs.columns.is_unique
train_motifs = core_train.join(extra_motifs)
test_motifs = core_test.join(extra_motifs)
labels = train.set_index('customer_id').target
print('Новых признаков', extra_motifs.shape[1])

Новых признаков 459


### Обучение на предыдущих месяцах

Общий бустинг использует 110 исходных признаков, отобранных внутри обучающего среза, и признаки последовательностей. Дополнительные модели повышают вес клиентов с похожей историей. Проверочные метки используются только при расчёте метрик. Веса смеси выбраны по результатам двух проверочных месяцев.

In [3]:
import hashlib
import os
import tempfile
from pathlib import Path

model_files = ['research/motif_model.py', 'research/window_models.py']
recipe = hashlib.sha256(b'motif-ensemble-v1')
for file in model_files:
    recipe.update(Path(file).read_bytes())
for package in ['numpy', 'pandas', 'scikit-learn']:
    recipe.update(version(package).encode())

def train_stage(stage, train_ids, query, baseline):
    fit = train_motifs.loc[train_ids]
    target = labels.loc[train_ids]
    chosen = list(select_by_gain(core_train.loc[train_ids], target, 110))
    feature_names = chosen + list(extra_motifs.columns)
    digest = recipe.copy()
    digest.update(repr(feature_names).encode())
    for obj in [fit, target, query]:
        digest.update(pd.util.hash_pandas_object(obj, index=True).to_numpy().tobytes())
    key = 'motifs_' + stage + '_' + digest.hexdigest()[:24]
    pooled = cached_forecast(key + '_general', query.index, lambda:
        forecast(fit[feature_names], target, query[feature_names]))
    local = cached_forecast(key + '_experts', query.index, lambda:
        motif_experts(fit, target, query, chosen))
    return .25 * baseline + .25 * pooled + .5 * local

def save_csv(frame, dest):
    with tempfile.NamedTemporaryFile(mode='w', dir=Path(dest).parent, delete=False) as tmp:
        frame.to_csv(tmp, index=False)
    os.replace(tmp.name, dest)

In [4]:
records = []
for date in sorted(train.scoring_date.unique())[1:]:
    before = train.scoring_date < date
    held = train.scoring_date == date
    ids = train.customer_id.loc[held]
    stage = str(date.date())
    baseline = score_stage(stage, before, ids, transition_train.loc[held], full_train.loc[held])['combined_regional']
    scores = train_stage(stage, train.customer_id.loc[before], train_motifs.loc[ids], baseline)
    observed = labels.loc[ids]
    k = int(np.ceil(.2 * len(observed)))
    for name, pred in [('previous', baseline), ('motifs', scores)]:
        top = np.argsort(-pred, kind='stable')[:k]
        buyers = int(observed.iloc[top].sum())
        records.append(dict(date=stage, model=name, ap20=ap20(observed, pred), buyers_in_top_5000=buyers))
    save_csv(pd.DataFrame(records), 'validation_motifs.csv')
print(pd.DataFrame(records).to_string(index=False))

      date    model     ap20  buyers_in_top_5000
2022-02-01 previous 0.760016                4114
2022-02-01   motifs 0.774831                4159
2022-03-01 previous 0.770113                4142
2022-03-01   motifs 0.782566                4169


### Прогноз для Kaggle

Для апрельского среза используем все 75 000 размеченных клиентов. Записываем строки в порядке образца соревнования, затем заново читаем файл и проверяем идентификаторы и прогнозы.

In [5]:
test_reference = pd.read_csv('submission_groups.csv').set_index('customer_id').score.loc[test.customer_id]
april = train_stage('april', train.customer_id, test_motifs, test_reference.to_numpy())
prediction = sample[['customer_id']].merge(pd.Series(april, index=test_motifs.index, name='score'),
                                         left_on='customer_id', right_index=True, validate='one_to_one')
assert prediction.customer_id.equals(sample.customer_id)
assert len(prediction) == len(test) and set(prediction.customer_id) == set(test.customer_id)
assert prediction.score.between(0, 1).all() and prediction.score.nunique() > 1
save_csv(prediction, 'submission_motifs.csv')
pd.testing.assert_frame_equal(prediction, pd.read_csv('submission_motifs.csv'))
print(len(prediction), 'клиентов в submission_motifs.csv')

25000 клиентов в submission_motifs.csv
